# Stage 0: walking skeleton (local machine)

The whole pipeline, end to end, on 20 photos: cut native-resolution tiles, describe them with
DINOv3 ViT-B/16, train a small gated-attention head for 50 steps, score the 8 held-back photos and
write a run record. The scores mean nothing. Stage 0 is done when this run and the Kaggle run give
**the same predicted labels and scores, with probabilities within 1e-3** (Stage 0).

The code lives in `src/skeleton.py`; this notebook only calls it.

In [ ]:
from src.config import load_config, path_for
from src.skeleton import compare_runs, run

CONFIG = "configs/skeleton.yaml"
config = load_config(CONFIG)
runs = path_for(config, "runs")

## 1. Run on this machine

A run name can only be used once (`RunRecorder` refuses to overwrite). Commit first, so the run
record holds a clean code version.

In [ ]:
metrics = run(CONFIG, "stage0_skeleton_halo")
metrics

## 2. Run on Kaggle

From the terminal (the code dataset must match the current commit):

```
./scripts/kaggle_sync_code.sh
./scripts/kaggle_push.sh 00-skeleton
uv run kaggle kernels status aztrek/ricetein-00-skeleton
./scripts/kaggle_pull.sh 00-skeleton      # files runs/stage0_skeleton_kaggle/
```

## 3. Compare (the Stage 0 "done when" check)

In [ ]:
ok = compare_runs(runs / "stage0_skeleton_halo", runs / "stage0_skeleton_kaggle",
                  atol=config["skeleton"]["compare_atol"])
assert ok, "local and Kaggle runs disagree: see the FAIL lines above"
print("Stage 0 check passed.")